In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten, Dense, 
                                     Dropout, BatchNormalization, Input, 
                                     LeakyReLU, Reshape, Conv2DTranspose)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from tensorflow.keras import backend as K
from IPython.display import Image, display
import matplotlib.pyplot as plt
K.clear_session()

In [ ]:
# Constants
IMAGE_SIZE = (128, 128)
LATENT_DIM = 100
BATCH_SIZE = 14
EPOCHS = 18
LEARNING_RATE_CNN = 0.001
LEARNING_RATE_GAN = 0.0002
CHECKPOINT_PATH = "./training_checkpoints"

# Prepare directories for checkpoints
os.makedirs(CHECKPOINT_PATH, exist_ok=True)

In [ ]:
# Dataset Path
DATASET_PATH = "/kaggle/input/signature-verification-dataset/extract"  
# This overall work project was run, trained and test on Kaggle Notebook Cloud as it can suffice with its Free CPU.


In [ ]:
#-------------------------------------
### 1. Data Loading and Preprocessing-
#-------------------------------------
def load_dataset(dataset_path):
    images, labels = [], []
    for folder_name in tqdm(os.listdir(dataset_path), desc="Loading dataset"):
        folder_path = os.path.join(dataset_path, folder_name)
        if not os.path.isdir(folder_path):  # Skip non-folder files
            continue
        is_forged = "_forg" in folder_name
        label = 0 if is_forged else 1  # 0: Forged, 1: Genuine
        for filename in os.listdir(folder_path):
            if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                img_path = os.path.join(folder_path, filename)
                img = load_img(img_path, target_size=IMAGE_SIZE)
                img_array = img_to_array(img) / 255.0
                images.append(img_array)
                labels.append(label)
    return np.array(images), np.array(labels)

print("Loading the dataset...")
X, y = load_dataset(DATASET_PATH)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Dataset loaded. Training samples: {len(X_train)}, Testing samples: {len(X_test)}")

In [ ]:
#-----------------------------------------------------------
### 2. GAN Components: Generator, Discriminator, and GAN   -
#-----------------------------------------------------------
def build_generator(latent_dim):
    model = Sequential([
        Dense(128 * 16 * 16, activation='relu', input_dim=latent_dim),
        Reshape((16, 16, 128)),
        Conv2DTranspose(128, (4, 4), strides=(2, 2), padding='same', activation='relu'),
        BatchNormalization(),
        Conv2DTranspose(64, (4, 4), strides=(2, 2), padding='same', activation='relu'),
        BatchNormalization(),
        Conv2DTranspose(32, (4, 4), strides=(2, 2), padding='same', activation='relu'),
        BatchNormalization(),
        Conv2D(3, (7, 7), activation='sigmoid', padding='same')  # Final output
    ])
    return model

def build_discriminator(input_shape):
    model = Sequential([
        Conv2D(64, (3, 3), strides=(2, 2), padding='same', input_shape=input_shape),
        LeakyReLU(alpha=0.2),
        Dropout(0.4),
        Conv2D(128, (3, 3), strides=(2, 2), padding='same'),
        LeakyReLU(alpha=0.2),
        Dropout(0.4),
        Flatten(),
        Dense(1, activation='sigmoid')
    ])
    return model

def build_gan(generator, discriminator):
    discriminator.trainable = False
    gan_input = Input(shape=(LATENT_DIM,))
    fake_image = generator(gan_input)
    gan_output = discriminator(fake_image)
    gan = Model(gan_input, gan_output)
    return gan

generator = build_generator(LATENT_DIM)
discriminator = build_discriminator((128, 128, 3))
gan = build_gan(generator, discriminator)

discriminator.compile(optimizer=Adam(learning_rate=LEARNING_RATE_GAN), loss='binary_crossentropy', metrics=['accuracy'])
gan.compile(optimizer=Adam(learning_rate=LEARNING_RATE_GAN), loss='binary_crossentropy')

In [ ]:
#---------------------
### 3. CNN Model     #
#--------------------
def build_cnn(input_shape):
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        MaxPooling2D((2, 2)),
        BatchNormalization(),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        BatchNormalization(),
        Conv2D(128, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    return model

cnn_model = build_cnn(input_shape=(128, 128, 3))
cnn_model.compile(optimizer=Adam(learning_rate=LEARNING_RATE_CNN), loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
#-----------------------------------------------------
### 4. Training Loop for GAN and Testing Loop for CNN #
#-----------------------------------------------------

#initialization of plot variables
generator_losses = []
discriminator_losses = []
discriminator_accuracies = []
cnn_losses = []
cnn_accuracies = []

def train_gan_test_cnn(generator, discriminator, gan, cnn, epochs, batch_size, latent_dim, X_train, y_train):
    real_labels = np.ones((batch_size, 1)) * 0.9  # Label smoothing
    fake_labels = np.zeros((batch_size, 1))
    half_batch = batch_size // 2

    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")

        # GAN Training Phase
        for _ in range(batch_size):
            for _ in range(2):  # Update discriminator twice
                noise = np.random.normal(0, 1, (batch_size, latent_dim))
                generated_images = generator.predict(noise)
                idx = np.random.randint(0, X_train.shape[0], half_batch)
                real_images = X_train[idx] + np.random.normal(0, 0.05, X_train[idx].shape)
                d_loss_real = discriminator.train_on_batch(real_images, real_labels[:half_batch])
                d_loss_fake = discriminator.train_on_batch(generated_images[:half_batch], fake_labels[:half_batch])
                d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

            g_loss = gan.train_on_batch(noise, real_labels)
            
        #GAN metrics
        generator_losses.append(g_loss)
        discriminator_losses.append(d_loss[0])
        discriminator_accuracies.append(d_loss[1])

        print(f"GAN - D Loss: {d_loss[0]:.4f}, D Accuracy: {d_loss[1]:.4f}, G Loss: {g_loss[0]:.4f}")

        # CNN Testing Phase
        noise = np.random.normal(0, 1, (batch_size, latent_dim))
        fake_images = generator.predict(noise)
        fake_labels_cnn = np.zeros((batch_size, 1))
        augmented_X = np.concatenate([X_train[:batch_size], fake_images])
        augmented_y = np.concatenate([y_train[:batch_size], fake_labels_cnn.flatten()])
        cnn_loss = cnn.train_on_batch(np.array(augmented_X), np.array(augmented_y))

        #CNN metrics
        cnn_losses.append(cnn_loss[0])
        cnn_accuracies.append(cnn_loss[1])
        
        print(f"CNN - Loss: {cnn_loss[0]:.4f}, Accuracy: {cnn_loss[1]:.4f}")

        train_gan_test_cnn(generator, discriminator, gan, cnn_model, EPOCHS, BATCH_SIZE, LATENT_DIM, X_train, y_train)

       
            

In [ ]:
if (epoch + 1) % 1 == 0:
            gan_plot_filename, cnn_plot_filename = save_loss_and_accuracy_plots(
                generator_losses, discriminator_losses, discriminator_accuracies, cnn_losses, cnn_accuracies, epoch)
            display(Image(gan_plot_filename))
            display(Image(cnn_plot_filename))
# 5. VISUALIZATION PART                             
def save_loss_and_accuracy_plots(generator_losses, discriminator_losses, discriminator_accuracies, cnn_losses, cnn_accuracies, epoch):
    if len(cnn_losses) == 0 or len(cnn_accuracies) == 0:
        print("No CNN data to plot. Skipping...")
        return None, None

    # Adjust the x-axis to match the data length
    epochs_range = range(1, len(cnn_losses) + 1)

    # Create a figure for GAN Loss and Discriminator Accuracy
    plt.figure(figsize=(24, 12))

    # Plot Losses for Generator and Discriminator
    plt.subplot(1, 2, 1)
    plt.plot(range(1, len(generator_losses) + 1), generator_losses, label='Generator Loss', color='blue')
    plt.plot(range(1, len(discriminator_losses) + 1), discriminator_losses, label='Discriminator Loss', color='red')
    plt.title(f'Epoch {epoch + 1} - GAN Losses')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    # Plot Accuracies for Discriminator
    plt.subplot(1, 2, 2)
    plt.plot(range(1, len(discriminator_accuracies) + 1), discriminator_accuracies, label='Discriminator Accuracy', color='green')
    plt.title(f'Epoch {epoch + 1} - GAN Discriminator Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    gan_plot_filename = f'gan_loss_and_accuracy_epoch_{epoch + 1}.png'
    plt.tight_layout()
    plt.savefig(gan_plot_filename)
    plt.close()

    # Create a figure for CNN Loss and Accuracy
    plt.figure(figsize=(24, 12))

    # Plot CNN Loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, cnn_losses, label='CNN Loss', color='red')
    plt.title('CNN Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    # Plot CNN Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, cnn_accuracies, label='CNN Accuracy', color='green')
    plt.title('CNN Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    cnn_plot_filename = f'cnn_loss_and_accuracy_epoch_{epoch + 1}.png'
    plt.tight_layout()
    plt.savefig(cnn_plot_filename)
    plt.close()

    return gan_plot_filename, cnn_plot_filename


#def save_generated_images(generator, latent_dim, epoch):
    #noise = np.random.normal(0, 1, (16, latent_dim))
   # generated_images = generator.predict(noise)
   # plt.figure(figsize=(10, 10))
    #for i in range(16):
     #   plt.subplot(4, 4, i + 1)
      #  plt.imshow(generated_images[i])
      #  plt.axis('off')
   # plt.tight_layout()
   # plt.savefig(f"generated_images_epoch_{epoch}.png")
   # plt.close()